<a href="https://colab.research.google.com/github/9terry-student/Transformer-based_LLM_from_scratch/blob/main/Transformer_from_numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Attention

In [33]:
import numpy as np
def softmax(x):
  a=np.exp(x)/np.exp(x).sum(axis=-1,keepdims=True)
  return a
def scaled_dot_product_attention(Q,K,V,mask=None):
  d_k=Q.shape[-1]
  weights=(Q@K.T)/np.sqrt(d_k)
  if mask is not None:
    mask=np.triu(np.full((seq_len,seq_len),-np.inf),k=1)
    weights+=mask
  weights=softmax(weights)
  output=weights@V
  return output,weights

In [34]:
# test
import numpy as np
np.random.seed(42)
seq_len=4
d_k=8

Q=np.random.randn(seq_len,d_k)
K=np.random.randn(seq_len,d_k)
V=np.random.randn(seq_len,d_k)

output,weights=scaled_dot_product_attention(Q,K,V)

print('weights 합: ',weights.sum(axis=-1).round(3))     # 1.0
print("output shape: ",output.shape)      # (4,8)

weights 합:  [1. 1. 1. 1.]
output shape:  (4, 8)


# 2. Multi-head attention

In [47]:
def multi_head_attention(Q,K,V,num_heads,mask=None):
  d_model=Q.shape[-1]
  d_k=d_model//num_heads

  W_Q=[np.random.randn(d_model,d_k) for _ in range(num_heads)]
  W_K=[np.random.randn(d_model,d_k) for _ in range(num_heads)]
  W_V=[np.random.randn(d_model,d_k) for _ in range(num_heads)]
  W_O=np.random.randn(d_model,d_model)

  heads=[]
  for i in range(num_heads):
    Q_i=Q@W_Q[i]
    K_i=K@W_K[i]
    V_i=V@W_V[i]
    head_i,_=scaled_dot_product_attention(Q_i,K_i,V_i)
    heads.append(head_i)

  heads=np.concatenate(heads,axis=-1)
  output=heads@W_O

  return output

In [48]:
# test
np.random.seed(42)
seq_len=4
d_model=16
num_heads=2

Q=np.random.randn(seq_len,d_model)
K=np.random.randn(seq_len,d_model)
V=np.random.randn(seq_len,d_model)

output=multi_head_attention(Q,K,V,num_heads)
print('output shape: ',output.shape)      # (4,16)

output shape:  (4, 16)


# 3. FeedForwardNetwork

In [49]:
def ReLU(x):
  return np.maximum(0,x)

def FFN(x):
  W1=np.random.randn(d_model,d_ff)
  W2=np.random.randn(d_ff,d_model)
  b1=np.zeros(d_ff)
  b2=np.zeros(d_model)
  output=ReLU(x@W1+b1)@W2+b2
  return output

In [50]:
# test
d_model=16
d_ff=64
seq_len=4

x=np.random.randn(seq_len,d_model)
output=FFN(x)
print('output shape: ',output.shape)      # (4,16)

output shape:  (4, 16)


# 4. LayerNorm

In [51]:
def LayerNorm(x):
  p1=np.ones(d_model)
  p2=np.zeros(d_model)
  output=((x-np.mean(x,axis=-1,keepdims=True))/(np.std(x,axis=-1,keepdims=True)+10**-9))*p1+p2
  return output

In [52]:
# test
x=np.random.randn(seq_len,d_model)
output=LayerNorm(x)

print('output shape: ',output.shape)      # (4,16)
print('mean: ',output.mean(axis=-1).round(3))     # 0~
print('std: ',output.std(axis=-1).round(3))      # 1~

output shape:  (4, 16)
mean:  [0. 0. 0. 0.]
std:  [1. 1. 1. 1.]


# 5. Position Encoding

In [53]:
def PE(pos,i):
  if i%2==0:
    return np.sin(pos/10**(4*(i/d_model)))
  return np.cos(pos/10*(4**((i-1)/d_model)))

In [54]:
# test
pe=np.array([[PE(pos,i) for i in range(d_model)] for pos in range(seq_len)])

print('PE shape: ',pe.shape)      # (4,16)

PE shape:  (4, 16)


# 6. Encoder

In [55]:
def Encoder(x):
  output=LayerNorm(x+multi_head_attention(x,x,x,num_heads))
  output=LayerNorm(output+FFN(output))
  return output

In [56]:
# test
x=np.random.randn(seq_len,d_model)
x+=pe
output=Encoder(x)

print('output shape: ',output.shape)      # (4,16)

output shape:  (4, 16)


# 7. Decoder

In [57]:
def Decoder(x,encoder_output):
  output=LayerNorm(x+multi_head_attention(x,x,x,num_heads,mask=True))
  output=LayerNorm(output+multi_head_attention(output,encoder_output,encoder_output,num_heads))
  output=LayerNorm(output+FFN(output))
  return output

In [58]:
# test1
np.random.seed(42)
seq_len=4
d_model=16
num_heads=2
d_ff=64

x=np.random.randn(seq_len,d_model)
pe=np.array([[PE(pos,i) for i in range(d_model)] for pos in range(seq_len)])

encoder_output=Encoder(x+pe)
decoder_output=Decoder(x+pe,encoder_output)

print('decoder output shape: ',decoder_output.shape)      # (4,16)

decoder output shape:  (4, 16)


In [59]:
# test2
Q=np.random.randn(seq_len,d_k)
K=np.random.randn(seq_len,d_k)
V=np.random.randn(seq_len,d_k)
_,weights_masked=scaled_dot_product_attention(Q, K, V, mask=True)
_,weights_normal=scaled_dot_product_attention(Q, K, V)

print("masked weights:\n",weights_masked.round(3))      # 위삼각형: 0
print("normal weights:\n",weights_normal.round(3))      # 0보다 큼
print("masked weights 합:",weights_masked.sum(axis=-1).round(3))      # [1. 1. 1. 1.]
print("normal weights 합:",weights_normal.sum(axis=-1).round(3))      # [1. 1. 1. 1.]

masked weights:
 [[1.    0.    0.    0.   ]
 [0.585 0.415 0.    0.   ]
 [0.206 0.185 0.61  0.   ]
 [0.136 0.045 0.724 0.096]]
normal weights:
 [[0.3   0.132 0.094 0.474]
 [0.177 0.125 0.1   0.598]
 [0.197 0.177 0.585 0.04 ]
 [0.136 0.045 0.724 0.096]]
masked weights 합: [1. 1. 1. 1.]
normal weights 합: [1. 1. 1. 1.]
